# Dependencies

In [241]:
import copy
import inspect
from rapidfuzz import process, fuzz
import json
from typing import Any, Callable
from datetime import date

# Fake DataBase for CUZ

In [242]:
class FakeDB():
    def __init__(self, data: dict[str, list[dict[str, Any]]]):
        self.data = data

    def query(self, table: str, predicate: Callable) -> list:
        return [
            row for row in self.data.get(table, [])
            if predicate(row)
        ]

    def get_n_closest(self, table: str, n: int, metric: Callable) -> list:
        return sorted(self.data.get(table, []), key=metric)[:n]

    # --- Restaurant helpers ---

    def get_restaurants_by_city(self, city: str) -> list:
        return self.query("restaurants", lambda r: r["location"]["city"].lower() == city.lower())

    def get_restaurants_by_cuisine(self, cuisine: str) -> list:
        return self.query("restaurants", lambda r: r["cuisine"].lower() == cuisine.lower())

    def get_restaurants_with_min_seats(self, min_seats: int) -> list:
        return self.query("restaurants", lambda r: r["seats"] >= min_seats)

    def get_restaurant_by_name(self, name: str) -> list:
        return self.query("restaurants", lambda r: r["name"].lower() == name.lower())

    def suggest_restaurants_for_visit(self, city: str, party_size: int = 1) -> list:
        """
        Given a city, and party size, return viable restaurants.
        Filters by city and minimum seat count.
        """
        return self.query(
            "restaurants",
            lambda r: r["location"]["city"].lower() == city.lower()
                      and r["seats"] >= party_size
        )

    def autocomplete_restaurant_name(self, partial: str) -> list[str]:
        """
        Suggest restaurant names matching a partial string.
        Useful after location is known to guide name_add.
        """
        return [
            r["name"] for r in self.data.get("restaurants", [])
            if partial.lower() in r["name"].lower()
        ]


data = {
    "restaurants": [
        {
            "location": {"country": "Sweden", "city": "Gothenburg", "street": "Kungsgatan 1"},
            "name": "Kebab123", "seats": 10, "cuisine": "Kebab"
        },
        {
            "location": {"country": "Sweden", "city": "Gothenburg", "street": "Avenyn 10"},
            "name": "Pasta Palace", "seats": 25, "cuisine": "Italian"
        },
        {
            "location": {"country": "Sweden", "city": "Stockholm", "street": "Drottninggatan 5"},
            "name": "Sushi Zen", "seats": 15, "cuisine": "Japanese"
        },
        {
            "location": {"country": "Sweden", "city": "Malmo", "street": "Stortorget 2"},
            "name": "The Green Garden", "seats": 40, "cuisine": "Vegetarian"
        },
        {
            "location": {"country": "Sweden", "city": "Gothenburg", "street": "Linnégatan 15"},
            "name": "Burger Joint", "seats": 20, "cuisine": "American"
        },
        {
            "location": {"country": "Sweden", "city": "Uppsala", "street": "Svartbäcksgatan 1"},
            "name": "Viking Feast", "seats": 50, "cuisine": "Nordic"
        }
    ]
}

db = FakeDB(data)

# Frame Definitions

In [410]:
class Frame:
    def __init__(self, expected_answer: list[str], 
                 field_prompts: dict[str, str], field_expected_answers: dict[str, list[str]], parent = None):
        self.name = type(self).__name__
        self.expected_answer = expected_answer
        self.completed = False
        self.field_prompts = field_prompts
        self.field_expected_answers = field_expected_answers
        self.parent = parent
        self.flat_field_answers = {
                answer: field
                for field, answers in field_expected_answers.items()
                for answer in answers
        }
        
    @property
    def fields(self):
        cls = type(self)
        members = inspect.getmembers(cls)
        return {
            name: value for name, value in members
            if not inspect.isroutine(value)
            and name in cls.__dict__
            and value.startswith(f"{cls.__name__}.")
            and not name.startswith('_')
        }
        
    @property
    def actions(self):
        cls = type(self)
        members = inspect.getmembers(cls, predicate=inspect.isfunction)
        return {
            name : func for name, func 
            in members if func.__qualname__.startswith(f"{cls.__name__}.")
            if not name.startswith('__')
        }

    @property
    def action_tokens(self):
        return {
            action: {k: None for k in inspect.signature(fn).parameters.keys()}
            for action, fn in self.actions.items()
        }

    def check_completed(self):
        for name, value in self.fields.items():
            if value is None:
                return False
            if isinstance(value, Frame) and not value.completed:
                return False
        self.completed = True
        return True
    
    def get_field_prompt(self, field: str):
        return self.field_prompts[field]

    def get_field_expected_answer(self, field: str):
        return self.field_expected_answers[field]
        
# Implementations

class Date(Frame):
    def __init__(self, year=None, month=None, day=None, parent=None):
        self.year = year
        self.month = month
        self.day = day
        super().__init__(
            expected_answer=["ON THE _"],
            field_prompts={
                "year": "WHAT YEAR?",
                "month": "WHAT MONTH?",
                "day": "WHAT DAY?"
            },
            field_expected_answers={
                "year_add": ["THE YEAR IS _",],
                "month_add": ["THE MONTH IS _"],
                "day_add": ["ON THE _", "THE DAY IS _"]
            },
            parent=parent
        )

    def year_add(self, year):
        self.year = year
        self.check_completed()
        return self

    def month_add(self, month):
        self.month = month
        self.check_completed()
        return self

    def day_add(self, day):
        self.day = day
        self.check_completed()
        return self


class Duration(Frame):
    def __init__(self, time=None, parent=None):
        self.time = time
        super().__init__(
            expected_answer=["I WANT TO STAY FOR _ _", "I WANT TO SPEND _ _ OF TIME"],
            field_prompts={
                "time": "HOW LONG DO YOU WANT TO STAY?"
            },
            field_expected_answers={
                "time_add": ["I WANT TO STAY FOR _ _", "I WANT TO SPEND _ _ OF TIME"]
            },
            parent=parent
        )

    def time_add(self, time):
        self.time = time
        self.check_completed()
        return self

class Location(Frame):
    def __init__(self, country=None, city=None, street=None, parent=None, query_type="restaurants"):
        self.country = country
        self.city = city
        self.street = street
        self.db = db
        self.query_type = query_type
        super().__init__(
            expected_answer=["I WANT TO GO TO _"],
            field_prompts={
                "country": "WHICH COUNTRY?\n" + self.country_suggest(),
                "city": "WHICH CITY?\n" + self.city_suggest(),
                "street": "WHICH STREET?\n" + self.street_suggest()
            },
            field_expected_answers={
                "country_add": ["THE COUNTRY IS _", "COUNTRY NAME IS _"],
                "city_add": ["THE CITY IS _", "CITY NAME IS _"],
                "street_add": ["STREET NAME IS _", "THE STREET IS _"]
            },
            parent=parent
        )

    def country_suggest(self) -> str:
        results = self.db.query(
            self.query_type,
            lambda r: (self.city is None   or r["location"]["city"].lower()   == self.city.lower()) and
                      (self.street is None or r["location"]["street"].lower()  == self.street.lower())
        )
        options = sorted({r["location"]["country"] for r in results})
        return f"(options: {', '.join(options)})" if options else ""

    def city_suggest(self) -> str:
        results = self.db.query(
            self.query_type,
            lambda r: (self.country is None or r["location"]["country"].lower() == self.country.lower()) and
                      (self.street is None  or r["location"]["street"].lower()  == self.street.lower())
        )
        options = sorted({r["location"]["city"] for r in results})
        return f"(options: {', '.join(options)})" if options else ""

    def street_suggest(self) -> str:
        results = self.db.query(
            self.query_type,
            lambda r: (self.country is None or r["location"]["country"].lower() == self.country.lower()) and
                      (self.city is None    or r["location"]["city"].lower()    == self.city.lower())
        )
        options = sorted({r["location"]["street"] for r in results})
        return f"(options: {', '.join(options)})" if options else ""

    def country_add(self, country):
        self.country = country
        self.field_prompts["city"]   = "WHICH CITY?\n"   + self.city_suggest()
        self.field_prompts["street"] = "WHICH STREET?\n" + self.street_suggest()
        self.check_completed()
        return self

    def city_add(self, city):
        self.city = city
        self.field_prompts["country"] = "WHICH COUNTRY?\n" + self.country_suggest()
        self.field_prompts["street"]  = "WHICH STREET?\n"  + self.street_suggest()
        self.check_completed()
        return self

    def street_add(self, street):
        self.street = street
        self.field_prompts["country"] = "WHICH COUNTRY?\n" + self.country_suggest()
        self.field_prompts["city"]    = "WHICH CITY?\n"    + self.city_suggest()
        self.check_completed()
        return self

class RestaurantVisit(Frame):
    def __init__(self, location = None, date = None, parent = None, name = None):
        self.location = location
        self.date = date
        self.name = name
        super().__init__(
            expected_answer=["I WANT TO GO TO _"],
            field_prompts={
                "location": "WHERE IS THE RESTAURANT LOCATED?",
                "date": "WHEN DO YOU WANT TO GO TO THE RESTAURANT?",
                "name": "WHAT IS THE NAME OF THE RESTAURANT?",
            },
            field_expected_answers={
                "location_add_country": ["ITS IN THE COUNTRY OF _", "THE COUNTRY IS _"],
                "location_add_city": ["ITS IN THE CITY OF _", "THE CITY IS _"],
                "location_add_street": ["ITS ON _ STREET", "THE STREET IS _"],
                "date_add_year": ["I WANT TO GO YEAR _"],
                "date_add_month": ["I WANT TO GO THERE IN _", "I WANT TO GO THERE THIS _"],
                "date_add_day": ["I WANT TO GO THERE ON THE _"],
                "date_add_nearby_weekday": [
                    "THIS MONDAY", "THIS TUESDAY", "THIS WEDNESDAY",
                    "THIS THURSDAY", "THIS FRIDAY", "THIS SATURDAY", "THIS SUNDAY"
                ],
                "name_add": ["IT IS CALLED _ ", "NAME IS _ "]
            },
            parent=parent
        )

    def location_add_country(self, country, parent):
        self.location = Location(country=country, parent=parent, query_type="restaurants")
        self.check_completed()
        return self.location

    def location_add_city(self, city, parent):
        self.location = Location(city=city, parent=parent, query_type="restaurants")
        self.check_completed()
        return self.location

    def location_add_street(self, street, parent):
        self.location = Location(street=street, parent=parent, query_type="restaurants")
        self.check_completed()
        return self.location

    def date_add_year(self, year, parent):
        self.date = Date(year=year, parent=parent)
        self.check_completed()
        return self.date

    def date_add_month(self, month, parent):
        self.date = Date(month=month, parent=parent)
        self.check_completed()
        return self.date

    def date_add_day(self, day, parent):
        self.date = Date(day=day, parent=parent)
        self.check_completed()
        return self.date

    def date_add_nearby_weekday(self, week_day):
        raise NotImplementedError()

    def name_add(self, name):
        self.name = name
        self.check_completed()
        return self

class WeatherForecast(Frame):
    def __init__(self, location = None, temperature = None, humidity = None, weather_type = None):
        self.location = location,
        self.temperature = temperature,
        self.humidity = humidity,
        self.weather_type = weather_type,
        super().__init__(
            expected_answer=["I WANT TO GO TO _"],
            field_prompts={
                "location": "WHERE IS THE RESTAURANT LOCATED?",
                "temperature": "WHEN DO YOU WANT TO GO TO THE RESTAURANT?",
                "humidity": "WHAT IS THE NAME OF THE RESTAURANT?",
            },
            field_expected_answers={
                "location_add_country": ["ITS IN THE COUNTRY OF _", "THE COUNTRY IS _"],
                "location_add_city": ["ITS IN THE CITY OF _", "THE CITY IS _"],
                "location_add_street": ["ITS ON _ STREET", "THE STREET IS _"],
                "date_add_year": ["I WANT TO GO YEAR _"],
                "date_add_month": ["I WANT TO GO THERE IN _", "I WANT TO GO THERE THIS _"],
                "date_add_day": ["I WANT TO GO THERE ON THE _"],
                "date_add_nearby_weekday": [
                    "THIS MONDAY", "THIS TUESDAY", "THIS WEDNESDAY",
                    "THIS THURSDAY", "THIS FRIDAY", "THIS SATURDAY", "THIS SUNDAY"
                ],
                "name_add": ["IT IS CALLED _ ", "NAME IS _ "]
            },
            parent=parent
        )

class Dialog(Frame):
    def __init__(self):
        self.restaurant_visit = None
        self.weather_forecast = None
        self.public_transport_trip = None
        super().__init__(
            expected_answer=[],
            field_prompts={
                "restaurant_visit": None,
                "weather_forecast": None,
                "public_transport_trip": None
            },
            field_expected_answers={
                "restaurant_visit_add":["I WANT TO BOOK A RESTAURANT"],
                "weather_forecast_add":["WHAT IS THE WEATHER?"],
                "public_transport_trip_add":["I WANT TO TRAVEL BY BUS"]
            },
            parent=None
        )

    def restaurant_visit_add(self, location, date, parent):
        self.restaurant_visit = RestaurantVisit(location, date, parent)
        self.check_completed()
        return self.restaurant_visit

    def weather_forecast_add(self, location, temperature, humidity, weather_type, parent):
        self.weather_forecast = WeatherForecast(location, temperature, humidity, weather_type, parent)
        self.check_completed()
        return self.weather_forecast

    def public_transport_trip_add(self, transport_type, departure_location, arrival_location, parent):
        self.public_transport_trip = PublicTransportTrip(transport_type, departure_location, arrival_location, parent)
        self.check_completed()
        return self.public_transport_trip

print(Dialog().fields.keys())

AttributeError: 'NoneType' object has no attribute 'startswith'

# Input Parser

In [409]:
class InputParser:
    def __init__(self, actionTemplates):
        self.actionTemplatesDict = actionTemplates
        self.quit_template = {
            "GOODBYE": "quit",
            "BYE": "quit",
            "QUIT": "quit",
            "EXIT": "quit"
        }

    def extract_named_slots(self, template, user_input, action, action_tokens):
        if action_tokens is None or action not in action_tokens:
            return {}
    
        template_words = template.upper().split()
        input_words = user_input.upper().split()
    
        blank_positions = [i for i, word in enumerate(template_words) if word == "_"]
        extracted_values = [input_words[i] for i in blank_positions if i < len(input_words)]
    
        slot_names = [k for k in action_tokens[action] if k != "parent"]
    
        # Only use template-as-value fallback for single-slot, no-blank actions
        # (e.g. date_add_nearby_weekday). Skip for actions whose slots are filled
        # by child frames (e.g. restaurant_visit_add, date_add_day with parent).
        if not blank_positions:
            if len(slot_names) == 1:
                return {slot_names[0]: template.upper().strip()}
            return {}  # slots filled by child frames, nothing to extract here
    
        # Use last N words of input for N blanks, rather than exact position matching
        extracted_values = input_words[-len(blank_positions):]
        slots = {slot_names[i]: extracted_values[i] for i in range(len(extracted_values))}
        return slots

    def update_templates(self, new_templates):
        self.actionTemplatesDict = new_templates
        self.actionTemplatesDict.update(self.quit_template)
        print(self.actionTemplatesDict)

    def match_intent(self, user_input):
        best = process.extractOne(
            user_input.upper(),
            self.actionTemplatesDict.keys(),
            scorer=fuzz.token_set_ratio
        )
        template, score, _ = best
        category = self.actionTemplatesDict[template]
        return template, category, score

    def getInputAction(self, user_input):
        template, action, score = self.match_intent(user_input)
        if score < 0.75:
            return False, "I DID NOT UNDERSTAND, TRY TO BE MORE PRECISE", ""
        else:
            return True, template, action

# Action Handler

In [393]:
class ContextManager:
    def __init__(self, initial_state):
        self.tokens = {}
        self.current_frame = initial_state
        self.available_action_tokens = initial_state.action_tokens

    def update_tokens(self, new_slots):
        """Merge newly extracted slots into the token store."""
        self.tokens.update(new_slots)

    def next_empty_field_name(self, frame):
        if frame is None:
            return None
        for name, value in frame.fields.items():
            if value is None:
                return name
        return None

    def next_frame_with_empty_field(self, frame):
        if frame is None:
            return None
        if frame.completed:
            return self.next_frame_with_empty_field(frame.parent)
        empty = self.next_empty_field_name(frame)
        if empty is not None:
            return frame
        return self.next_frame_with_empty_field(frame.parent)

    def perform_action(self, action):
        fn = self.current_frame.actions[action]
        sig = inspect.signature(fn)
        self.tokens["parent"] = self.current_frame
        args = {k: self.tokens.get(k) for k in sig.parameters.keys()}
        return fn(**args)

    def process_input_action(self, action, slots):
        """Accept slots so they're available before perform_action runs."""
        self.update_tokens(slots)
        result = self.perform_action(action)

        next_frame = self.next_frame_with_empty_field(result)

        if next_frame is None:
            return None, {}, {}

        next_empty_field_name = self.next_empty_field_name(next_frame)
        field_prompt = next_frame.get_field_prompt(next_empty_field_name)
        action_templates = next_frame.flat_field_answers
        self.available_action_tokens = next_frame.action_tokens
        self.current_frame = next_frame
        return field_prompt, action_templates, self.available_action_tokens

    def print_state(self):
        """Debug print of current frame and token store."""
        print("\n--- STATE ---")
        print(f"Current frame : {type(self.current_frame).__name__}")
        print(f"Frame fields  : {self.current_frame.fields}")
        print(f"Completed     : {self.current_frame.completed}")
        print(f"Tokens        : {self.tokens}")
        print("-------------\n")

# Interaction Loop

In [394]:
dialog_frame = Dialog()
parser = InputParser(dialog_frame.flat_field_answers)
context_manager = ContextManager(dialog_frame)
available_action_tokens = None

print("HELLO, I AM CUZ, THE FRIENDLY ASSISTANT. WHAT CAN I HELP YOU WITH TODAY?\n")

while True:
    user_input = input(": ")
    ok, template, action = parser.getInputAction(user_input.upper())

    if (not ok):
        print(template)
        continue

    slots = parser.extract_named_slots(
        template, user_input, action, context_manager.available_action_tokens
    )
    print(slots)

    if action == "quit":
        print("Goodbye!")
        break

    prompt, action_templates, _ = context_manager.process_input_action(action, slots)
    context_manager.print_state()

    if prompt is None:
        print("All done! Here is what I have recorded:")
        context_manager.print_state()
        break

    print(prompt)
    parser.update_templates(action_templates)

HELLO, I AM CUZ, THE FRIENDLY ASSISTANT. WHAT CAN I HELP YOU WITH TODAY?



:  restaurant


{}


AttributeError: 'NoneType' object has no attribute 'restaurant_visit'